<a href="https://colab.research.google.com/github/ps-research/The-Language-of-AI-Liability/blob/main/Word_List.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ── Word List Frequency Analysis ──
# Run this after loading preprocessed_corpus.json
# Reports per-term frequencies for all word lists across all techniques
# Addresses reviewer concern: "report their frequency in the corpus"

import json
import re
from pathlib import Path
from collections import Counter, defaultdict

BASE_DIR = Path("/content/drive/MyDrive/ACL SRW TEXTS")
OUTPUT_DIR = BASE_DIR / "processed"

with open(OUTPUT_DIR / "preprocessed_corpus.json", "r", encoding="utf-8") as f:
    results = json.load(f)

LAW_ORDER = [
    "eu_ai_act", "china_genai", "south_korea",
    "ca_sb1047", "ca_sb53", "ca_sb942", "ca_ab2013",
    "co_sb205", "tx_traiga", "il_hb3773",
]

# Collect all obligation sentences
all_obligation_sents = []
for law_id in LAW_ORDER:
    for s in results[law_id]["sentences"]:
        if s["is_obligation"]:
            all_obligation_sents.append({
                "law_id": law_id,
                "sent_id": s["sent_id"],
                "text": s["text"],
            })

print(f"Total obligation sentences across corpus: {len(all_obligation_sents)}")
print()


# ═══════════════════════════════════════════════════════════════════
# 1. HEDGE TERMS (Technique 4)
# ═══════════════════════════════════════════════════════════════════

HEDGE_TERMS = [
    (r'\breasonable\b', "reasonable"),
    (r'\breasonably\b', "reasonably"),
    (r'\bappropriate\b', "appropriate"),
    (r'\bappropriately\b', "appropriately"),
    (r'\badequate\b', "adequate"),
    (r'\badequately\b', "adequately"),
    (r'\bsufficient\b', "sufficient"),
    (r'\bsufficiently\b', "sufficiently"),
    (r'\bmaterial\b', "material"),
    (r'\bmaterially\b', "materially"),
    (r'\bsignificant\b', "significant"),
    (r'\bsignificantly\b', "significantly"),
    (r'\bas\s+applicable\b', "as applicable"),
    (r'\bas\s+necessary\b', "as necessary"),
    (r'\bas\s+appropriate\b', "as appropriate"),
    (r'\bto\s+the\s+extent\s+practicable\b', "to the extent practicable"),
    (r'\bto\s+the\s+extent\s+possible\b', "to the extent possible"),
    (r'\bwhere\s+appropriate\b', "where appropriate"),
    (r'\bwhere\s+applicable\b', "where applicable"),
    (r'\bwhere\s+necessary\b', "where necessary"),
    (r'\bproportionate\b', "proportionate"),
    (r'\bsuitable\b', "suitable"),
    (r'\bfeasible\b', "feasible"),
]

print("=" * 72)
print("HEDGE TERM FREQUENCIES (across all obligation sentences)")
print("=" * 72)

hedge_freq = Counter()
hedge_by_law = defaultdict(Counter)

for sent in all_obligation_sents:
    for pattern, label in HEDGE_TERMS:
        matches = re.findall(pattern, sent["text"], re.IGNORECASE)
        if matches:
            hedge_freq[label] += len(matches)
            hedge_by_law[sent["law_id"]][label] += len(matches)

total_hedges = sum(hedge_freq.values())
print(f"\n{'Term':<30} {'Count':>6} {'% of all':>8}")
print("-" * 48)
for term, count in hedge_freq.most_common():
    print(f"  {term:<28} {count:>6} {count/total_hedges*100:>7.1f}%")
print(f"  {'TOTAL':<28} {total_hedges:>6}")

# Top 3 terms' share
top3 = hedge_freq.most_common(3)
top3_total = sum(c for _, c in top3)
print(f"\nTop 3 terms account for {top3_total/total_hedges*100:.1f}% of all hedge matches")
print()


# ═══════════════════════════════════════════════════════════════════
# 2. CONDITIONAL MARKERS (Technique 3)
# ═══════════════════════════════════════════════════════════════════

CONDITIONAL_TERMS = [
    (r'\bif\b', "if"),
    (r'\bunless\b', "unless"),
    (r'\bprovided\s+that\b', "provided that"),
    (r'\bexcept\s+(?:when|where|that|as)\b', "except when/where/that/as"),
    (r'\bto\s+the\s+extent\s+that\b', "to the extent that"),
    (r'\bsubject\s+to\b', "subject to"),
    (r'\bnotwithstanding\b', "notwithstanding"),
    (r'\bin\s+the\s+event\s+that\b', "in the event that"),
    (r'\bwhere\b', "where"),
    (r'\bin\s+(?:the\s+)?case\s+(?:of|that|where)\b', "in (the) case of/that/where"),
    (r'\bprovided\b', "provided"),
]

print("=" * 72)
print("CONDITIONAL MARKER FREQUENCIES (across all obligation sentences)")
print("=" * 72)

cond_freq = Counter()
for sent in all_obligation_sents:
    for pattern, label in CONDITIONAL_TERMS:
        matches = re.findall(pattern, sent["text"], re.IGNORECASE)
        if matches:
            cond_freq[label] += len(matches)

total_conds = sum(cond_freq.values())
print(f"\n{'Term':<35} {'Count':>6} {'% of all':>8}")
print("-" * 53)
for term, count in cond_freq.most_common():
    print(f"  {term:<33} {count:>6} {count/total_conds*100:>7.1f}%")
print(f"  {'TOTAL':<33} {total_conds:>6}")
print()


# ═══════════════════════════════════════════════════════════════════
# 3. TEMPORAL MARKERS (Technique 4)
# ═══════════════════════════════════════════════════════════════════

PRECISE_TEMPORAL_TERMS = [
    (r'\bwithin\s+\d+\s+(?:days?|hours?|months?|years?|weeks?|business\s+days?)\b',
     "within N days/hours/etc"),
    (r'\bno\s+later\s+than\b', "no later than"),
    (r'\bannually\b', "annually"),
    (r'\bquarterly\b', "quarterly"),
    (r'\bby\s+(?:January|February|March|April|May|June|July|August|'
     r'September|October|November|December)\s+\d+', "by [Month] [Day]"),
]

VAGUE_TEMPORAL_TERMS = [
    (r'\bpromptly\b', "promptly"),
    (r'\bin\s+a\s+timely\s+manner\b', "in a timely manner"),
    (r'\bas\s+soon\s+as\s+practicable\b', "as soon as practicable"),
    (r'\bas\s+soon\s+as\s+possible\b', "as soon as possible"),
    (r'\bperiodically\b', "periodically"),
    (r'\bwithout\s+(?:undue\s+)?delay\b', "without (undue) delay"),
    (r'\bregularly\b', "regularly"),
]

print("=" * 72)
print("TEMPORAL MARKER FREQUENCIES (across all obligation sentences)")
print("=" * 72)

print("\nPrecise temporal markers:")
prec_freq = Counter()
for sent in all_obligation_sents:
    for pattern, label in PRECISE_TEMPORAL_TERMS:
        matches = re.findall(pattern, sent["text"], re.IGNORECASE)
        if matches:
            prec_freq[label] += len(matches)

for term, count in prec_freq.most_common():
    print(f"  {term:<35} {count:>4}")
print(f"  {'TOTAL':<35} {sum(prec_freq.values()):>4}")

print("\nVague temporal markers:")
vague_freq = Counter()
for sent in all_obligation_sents:
    for pattern, label in VAGUE_TEMPORAL_TERMS:
        matches = re.findall(pattern, sent["text"], re.IGNORECASE)
        if matches:
            vague_freq[label] += len(matches)

for term, count in vague_freq.most_common():
    print(f"  {term:<35} {count:>4}")
print(f"  {'TOTAL':<35} {sum(vague_freq.values()):>4}")
print()


# ═══════════════════════════════════════════════════════════════════
# 4. FALSE POSITIVE SAMPLING — 20 random matches per word list
# ═══════════════════════════════════════════════════════════════════

import random
random.seed(42)

def sample_matches(sentences, pattern_list, list_name, n_samples=20):
    """Sample n random matches with context for manual review."""
    all_matches = []
    for sent in sentences:
        for pattern, label in pattern_list:
            if re.search(pattern, sent["text"], re.IGNORECASE):
                all_matches.append({
                    "law_id": sent["law_id"],
                    "sent_id": sent["sent_id"],
                    "term": label,
                    "text": sent["text"][:250],
                })

    sampled = random.sample(all_matches, min(n_samples, len(all_matches)))

    print(f"\n{'=' * 72}")
    print(f"FALSE POSITIVE SAMPLE: {list_name} ({len(sampled)} of {len(all_matches)} matches)")
    print(f"{'=' * 72}")
    for i, m in enumerate(sampled, 1):
        print(f"\n  [{i}] {m['law_id']} sent#{m['sent_id']} — term: '{m['term']}'")
        print(f"      {m['text']}")

    return sampled


hedge_samples = sample_matches(
    all_obligation_sents, HEDGE_TERMS, "Hedge Terms", n_samples=20)

cond_samples = sample_matches(
    all_obligation_sents, CONDITIONAL_TERMS, "Conditional Markers", n_samples=20)

temporal_samples = sample_matches(
    all_obligation_sents,
    PRECISE_TEMPORAL_TERMS + VAGUE_TEMPORAL_TERMS,
    "Temporal Markers", n_samples=20)

Total obligation sentences across corpus: 1038

HEDGE TERM FREQUENCIES (across all obligation sentences)

Term                            Count % of all
------------------------------------------------
  appropriate                     133    27.8%
  where applicable                 47     9.8%
  reasonable                       47     9.8%
  reasonably                       39     8.2%
  as appropriate                   39     8.2%
  adequate                         23     4.8%
  as applicable                    23     4.8%
  materially                       20     4.2%
  where appropriate                18     3.8%
  sufficient                       16     3.3%
  as necessary                     13     2.7%
  proportionate                    11     2.3%
  where necessary                  10     2.1%
  feasible                          9     1.9%
  significant                       8     1.7%
  sufficiently                      5     1.0%
  material                          5     1.0%